<a href="https://colab.research.google.com/github/jano87/diplomado-MLunsupervised/blob/main/AllCategoriesFullDiscoveryMLC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# --- CELL 1: SETUP DISCOVERY DATABASE ---
from google.colab import drive
import sqlite3
import pandas as pd
import requests
import time
from datetime import datetime

drive.mount('/content/drive')
DB_PATH = '/content/drive/MyDrive/ml_category_discovery.db'

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS discovery_listings (
        id TEXT,
        title TEXT,
        category_id TEXT,
        category_name TEXT,
        price REAL,
        available_quantity INTEGER,
        condition TEXT,
        timestamp DATETIME
    )
''')
conn.commit()
print(f"Discovery Database ready at: {DB_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Discovery Database ready at: /content/drive/MyDrive/ml_category_discovery.db


In [7]:
# --- CELL 2: WIDE-NET CRAWLER ---
ROOT_CATEGORIES_TO_SCAN = 5   # How many main categories to check today
SUBCATS_PER_ROOT = 10         # How many subcategories to check per main category
TOP_ITEMS_PER_SUBCAT = 5      # How many top listings to sample per subcategory

def get_root_categories():
    try:
        url = "https://api.mercadolibre.com/sites/MLC/categories"
        res = requests.get(url)
        if res.status_code == 200:
            data = res.json()
            if isinstance(data, list):
                return data
        print(f"Warning: Failed to fetch root categories. Status: {res.status_code}")
    except Exception as e:
        print(f"Error fetching root categories: {e}")
    return []

def get_subcategories(parent_id):
    try:
        url = f"https://api.mercadolibre.com/categories/{parent_id}"
        res = requests.get(url)
        if res.status_code == 200:
            data = res.json()
            return data.get('children_categories', [])
    except Exception as e:
        print(f"Error fetching subcategories for {parent_id}: {e}")
    return []

def sample_top_listings(category_id, limit=5):
    try:
        url = f"https://api.mercadolibre.com/sites/MLC/search?category={category_id}&limit={limit}"
        res = requests.get(url)
        if res.status_code == 200:
            return res.json().get('results', [])
    except Exception as e:
        print(f"Error fetching items for category {category_id}: {e}")
    return []

print("Starting Wide-Net Discovery Crawl...")
current_time = datetime.now()
inserted_count = 0

# 1. Fetch Root Categories
roots = get_root_categories()[:ROOT_CATEGORIES_TO_SCAN]

for root in roots:
    print(f"Scanning Root: {root.get('name')} ({root.get('id')})")

    # 2. Fetch Subcategories for this Root
    subcats = get_subcategories(root.get('id'))[:SUBCATS_PER_ROOT]

    for subcat in subcats:
        # 3. Fetch Top Items for this Subcategory
        items = sample_top_listings(subcat.get('id'), limit=TOP_ITEMS_PER_SUBCAT)

        for item in items:
            if item.get('condition') == 'new':
                cursor.execute('''
                    INSERT INTO discovery_listings
                    (id, title, category_id, category_name, price, available_quantity, condition, timestamp)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                ''', (
                    item.get('id'),
                    item.get('title'),
                    subcat.get('id'),
                    subcat.get('name'),
                    item.get('price'),
                    item.get('available_quantity'),
                    item.get('condition'),
                    current_time
                ))
                inserted_count += 1

        # Brief pause to respect API rate limits
        time.sleep(0.2)

conn.commit()
print(f"\u2705 Crawl Complete: Logged {inserted_count} total top-performing items across {len(roots)} root categories at {current_time.strftime('%Y-%m-%d %H:%M:%S')}")

Starting Wide-Net Discovery Crawl...
✅ Crawl Complete: Logged 0 total top-performing items across 0 root categories at 2026-10-22 22:39:47


In [10]:
import requests

url = "https://api.mercadolibre.com/sites/MLC/categories"

print("--- Testing request without headers ---")
res1 = requests.get(url)
print(f"Status Code: {res1.status_code}")
print(f"Response Body: {res1.text}\n")

print("--- Testing request with User-Agent header ---")
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
res2 = requests.get(url, headers=headers)
print(f"Status Code: {res2.status_code}")
if res2.status_code == 200:
    print(f"Success! Categories found: {len(res2.json())}")
else:
    print(f"Response Body: {res2.text}")

--- Testing request without headers ---
Status Code: 403
Response Body: {"message":"At least one policy returned UNAUTHORIZED.","blocked_by":"PolicyAgent","status":403,"code":"PA_UNAUTHORIZED_RESULT_FROM_POLICIES"}


--- Testing request with User-Agent header ---
Status Code: 403
Response Body: {"status":403,"blocked_by":"PolicyAgent","message":"At least one policy returned UNAUTHORIZED.","code":"PA_UNAUTHORIZED_RESULT_FROM_POLICIES"}



In [8]:
# --- CELL 3: RANK CATEGORIES BY VELOCITY ---
def rank_categories_by_velocity():
    query = """
    WITH RankedData AS (
        SELECT
            id,
            category_id,
            category_name,
            price,
            available_quantity,
            timestamp,
            ROW_NUMBER() OVER(PARTITION BY id ORDER BY timestamp DESC) as rn
        FROM discovery_listings
    ),
    ItemVelocity AS (
        SELECT
            t1.category_id,
            t1.category_name,
            t1.id as item_id,
            (t2.available_quantity - t1.available_quantity) as units_sold,
            (t1.price * (t2.available_quantity - t1.available_quantity)) as revenue
        FROM RankedData t1
        JOIN RankedData t2 ON t1.id = t2.id
        WHERE t1.rn = 1 AND t2.rn = 2
        AND (t2.available_quantity - t1.available_quantity) > 0
    )
    -- Group the individual item velocities into category totals
    SELECT
        category_name,
        category_id,
        COUNT(DISTINCT item_id) as top_items_selling,
        SUM(units_sold) as total_units_sold_24h,
        SUM(revenue) as total_revenue_24h
    FROM ItemVelocity
    GROUP BY category_id, category_name
    ORDER BY total_revenue_24h DESC;
    """

    df = pd.read_sql_query(query, conn)
    return df

print("🔥 Most Profitable Niches (Last 24h Aggregation):")
heatmap_df = rank_categories_by_velocity()
display(heatmap_df.head(20))

🔥 Most Profitable Niches (Last 24h Aggregation):


,category_name,category_id,top_items_selling,total_units_sold_24h,total_revenue_24h
